## Bobot

### on importe les biblios :

In [1]:
import pandas as pd
import json 
import string 
import random 
import nltk
import numpy as np
from nltk.stem import WordNetLemmatizer
import tensorflow as tf
from tensorflow.keras import Sequential 
from tensorflow.keras.layers import Dense, Dropout 
nltk.download("punkt")
nltk.download('punkt_tab')
nltk.download("wordnet")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Utilisateur\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Utilisateur\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Utilisateur\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:
df_phrases = pd.read_json("C:\\Users\\Utilisateur\\Desktop\\Microsoft-chatbot-\\bobot\\hellos.json")

In [3]:
json_path = "C:\\Users\\Utilisateur\\Desktop\\Microsoft-chatbot-\\bobot\\hellos.json"

with open(json_path, "r", encoding = "utf-8") as f:
    data = json.load(f)

print(data["intents"][0])
print(len(data["intents"]))

{'tag': 'greeting', 'patterns': ['salut', 'bonjour', 'ça va?'], 'responses': ['hello', 'ça va et toi?']}
2


In [4]:
#what is in my .json ?
with open ("C:\\Users\\Utilisateur\\Desktop\\Microsoft-chatbot-\\bobot\\hellos.json", "r", encoding = "utf-8") as f: 
    print(f.read())

{
    "intents":[
        {"tag": "greeting",
        "patterns":["salut", "bonjour", "ça va?"],
        "responses":["hello", "ça va et toi?"]
        }, 
        {
        "tag": "age",
        "patterns":["Quel âge as-tu?", "C'est quand ton anniversaire?", "Quand es-tu né?"],
        "responses":[ "J'ai 25 ans", "Je suis né en 1996"]
        } 
    ]
}




In [5]:
with open("C:\\Users\\Utilisateur\\Desktop\\Microsoft-chatbot-\\bobot\\hellos.json", "r", encoding = "utf-8") as f: 
    data = json.load(f)

df_phrases = pd.json_normalize(data["intents"])
print(df_phrases)

        tag                                           patterns  \
0  greeting                           [salut, bonjour, ça va?]   
1       age  [Quel âge as-tu?, C'est quand ton anniversaire...   

                           responses  
0             [hello, ça va et toi?]  
1  [J'ai 25 ans, Je suis né en 1996]  


### séparation des données :

In [6]:
lemmatizer = WordNetLemmatizer()

In [7]:
words = []
classes = []
doc_X = []
doc_y = []

for intent in data["intents"]:
    for pattern in intent["patterns"]: 
        tokens = nltk.word_tokenize(pattern)
        words.extend(tokens)
        doc_X.append(pattern)
        doc_y.append(intent["tag"])

    if intent["tag"] not in classes: 
        classes.append(intent["tag"])

words = [lemmatizer.lemmatize(word.lower()) for word in words if word not in string.punctuation]

words = sorted(set(words))
classes = sorted(set(classes))

In [8]:
print(words)
print(classes)
print(doc_X)
print(doc_y)

['anniversaire', 'as-tu', 'bonjour', "c'est", 'es-tu', 'né', 'quand', 'quel', 'salut', 'ton', 'va', 'âge', 'ça']
['age', 'greeting']
['salut', 'bonjour', 'ça va?', 'Quel âge as-tu?', "C'est quand ton anniversaire?", 'Quand es-tu né?']
['greeting', 'greeting', 'greeting', 'age', 'age', 'age']


### Traitement des données (sac des mots / bag of words - bow)

In [9]:
training = []
out_empty = [0] * len(classes)

for idx, doc in enumerate(doc_X):
    bow = []
    text = lemmatizer.lemmatize(doc.lower())
    for word in words: 
        bow.append(1) if word in text else bow.append(0)

        output_row = list(out_empty)
        output_row[classes.index(doc_y[idx])] = 1

        training.append([bow, output_row])

random.shuffle(training)
training = np.array(training, dtype = object)

train_X = np.array(list(training[:, 0]))
train_y = np.array(list(training[:, 1]))

### Construction du réseau de neurones de Deep Learning - the model itself

In [10]:
input_shape = (len(train_X[0]),)
output_shape = len(train_y[0])
epochs = 200

In [11]:
model = Sequential()
model.add(Dense(128, input_shape = input_shape, activation = "relu"))
model.add(Dropout(0.5))
model.add(Dense(64, activation = "relu"))
model.add(Dropout(0.3))
model.add(Dense(output_shape, activation = "softmax"))

adam = tf.keras.optimizers.Adam(learning_rate = 0.01, decay = 1e-6)

model.compile(loss = 'categorical_crossentropy', optimizer = adam, metrics = ["accuracy"])

C:\Users\Utilisateur\anaconda3\envs\BobLeBot\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\Users\Utilisateur\anaconda3\envs\BobLeBot\Lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


In [12]:
print(model.summary())

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,178 (39.76 KB)

 Trainable params: 10,178 (39.76 KB)

 Non-trainable params: 0 (0.00 B)

None


In [13]:
# entraînement du modèle
model.fit(x = train_X, y = train_y, epochs = 200, verbose = 1)

Epoch 1/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.6667 - loss: 0.5823
Epoch 2/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 1.0000 - loss: 0.2665
Epoch 3/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 1.0000 - loss: 0.0810
Epoch 4/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 1.0000 - loss: 0.0141
Epoch 5/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 1.0000 - loss: 0.0024
Epoch 6/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 1.0000 - loss: 3.1350e-04
Epoch 7/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 1.0000 - loss: 1.5477e-04
Epoch 8/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 1.0000 - loss: 5.4261e-05
Epoch 9/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 1.0000 - loss: 8.9430e-06
Epoch 10/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 1.0000 - loss: 6.1126e-05
Epoch 11/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 1.0000 - loss: 6.7867e-06
Epoch 12/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step

### Création de l’application de chatbot (les fonctions pour utiliser le modèle)

In [14]:
def clean_text(text):
    tokens = nltk.word_tokenize(text)
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return tokens 

In [15]:
def bag_of_words(text, vocab): 
    tokens = clean_text(text) 
    bow = [0] * len(vocab)
    for w in tokens: 
        for idx, word in enumerate(vocab):  
            if word == w:
                bow[idx] = 1
    return np.array(bow)

In [16]:
def pred_class(text, vocab, labels): 
    bow = bag_of_words(text, vocab)
    result = model.predict(np.array([bow]))[0]
    thresh = 0.2
    y_pred = [[idx, res] for idx, res in enumerate(result) if res > thresh]

    y_pred.sort(key = lambda x: x[1], reverse = True)
    return_list = []
    for r in y_pred: 
        return_list.append(labels[r[0]])
    return return_list

In [17]:
def get_response(intents_list, intents_json):
    tag = intents_list[0]
    list_of_intents = intents_json["intents"]
    for i in list_of_intents: 
        if i["tag"] == tag: 
            result = random.choice(i["responses"])
            break
    return result

In [ ]:
while True: 
    message = input("")
    intents = pred_class(message, words, classes)
    result = get_response(intents, data)
    print(result)
    if message == ('stop'):
        break

 alors, le chatbot n'est plus un secret pour vous ? 


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
ça va et toi?
